# GLDS-525 Pipeline, Notebook 1: Setup, Download & Quality Control

**Study:** RR-23 Mouse Cerebellum Bulk RNA-Seq (NASA GeneLab OSD-525 / GLDS-525)  
**Organism:** *Mus musculus*  
**Genome:** Ensembl GRCm39 release 112 (+ ERCC92 spike-ins)

---

## 🎓 Tutorial Introduction: What is Bulk RNA-Seq?

Welcome! This is the first notebook in a four-part beginner-friendly tutorial that walks you through a complete **bulk RNA-sequencing (RNA-seq) analysis, taking you from raw data files all the way to a list of genes that change during spaceflight.

### What are we studying?
This dataset comes from NASA's **GeneLab** (dataset GLDS-525). Researchers sent mice to the International Space Station as part of the **RR-23 (Rodent Research-23)** mission, then examined gene expression in their cerebellum (a brain region important for balance and motor control). The goal is to find which genes are turned on or off by spaceflight compared to ground controls.

### What is RNA-seq?
Every cell in your body contains the same DNA, but different cells behave differently because they **express** (read and use) different genes. When a gene is active, the cell transcribes it into **RNA** molecules. RNA-seq measures how many RNA molecules are present for each gene , giving us a snapshot of which genes are "turned on" and how strongly.

**The pipeline in a nutshell:**
```
Raw sequence reads (FASTQ)
        ↓  Quality control & trimming (Notebook 1)
        ↓  Align reads to mouse genome (Notebook 2)
        ↓  Count reads per gene (Notebook 3)
        ↓  Find differentially expressed genes (Notebook 4)
```

---

## What This Notebook Does

| Step | Description | Why It Matters |
|---|---|---|
| **0** | System benchmark | Makes sure your computer can handle the workload |
| **1** | Tool installation check | Confirms all required software is installed |
| **2** | Project directory setup | Creates a tidy folder structure for all outputs |
| **3** | Download raw FASTQs | Gets the actual sequencing data from NASA GeneLab |
| **4** | Raw read QC with FastQC | Checks data quality before any processing |
| **5** | Aggregate QC with MultiQC | Combines QC reports for all 29 samples |
| **6** | Adapter trimming with TrimGalore | Removes sequencing artifacts |
| **7** | Post-trim FastQC + MultiQC | Verifies trimming improved quality |

---

## ⚙️ Things You Must Customize Before Running

> **Read this section carefully before executing any cells.**

The following settings in this notebook **must be edited to match your own computer and files**. They are marked with `# ← EDIT THIS` comments in the code cells.

| What to customize | Where | Default value |
|---|---|---|
| **Project directory path** | Section 2 | `~/GLDS525_project` |
| **Number of download workers** | Section 3 | `min(4, USABLE_CORES)` |
| **Runsheet CSV filename** | Section 3 | `GLDS-525_rna_seq_bulkRNASeq_v2_runsheet.csv` |
| **TEST_MODE** | Section 3 | `False` (set `True` to test with 1 sample first) |

---

## 📄 Required File: The Runsheet

Before running Section 3, you must download the **runsheet CSV** from GeneLab and place it in your project's `runsheet/` folder.

**File name:** `GLDS-525_rna_seq_bulkRNASeq_v2_runsheet.csv`  
**Download from:** https://genelab-data.ndc.nasa.gov/genelab/accession/GLDS-525/

The runsheet contains one row per sample with columns including sample name, group (spaceflight/ground control/vivarium), and download URLs for each FASTQ file. Section 3 reads this file automatically to build the download list.

**Example row:**
```
Sample Name: RR23_Cb_F1_Rep1_techrep1
read1_path:  https://genelab-data.../GLDS-525_...R1_raw.fastq.gz
read2_path:  https://genelab-data.../GLDS-525_...R2_raw.fastq.gz
Factor Value[Spaceflight]: Space Flight
```

---

## Prerequisites: Software Installation

Install all tools via conda before running this notebook:

```bash
conda create -n glds525_upstream -c bioconda -c conda-forge \
    fastqc=0.12.1 trim-galore=0.6.10 multiqc=1.26 \
    star=2.7.11b rsem=1.3.3 samtools=1.21 \
    rseqc wget pigz python=3.10
conda activate glds525_upstream

# Register the environment as a Jupyter kernel
python -m ipykernel install --user --name glds525_upstream --display-name "GLDS-525 Upstream"
```

> 💡 **VS Code users:** After running `python -m ipykernel install` above, reload VS Code and select the **"GLDS-525 Upstream"** kernel in the top-right kernel picker. If you don't see it, install the Jupyter extension and run `pip install ipykernel` inside the conda environment.

> **Note:** The downstream DEG analysis (Notebook 4) uses a separate Python environment
> with PyDESeq2. Keep the two environments separate to avoid conflicts.

## Pipeline File Structure
```
GLDS525_project/
├── runsheet/                  ← place runsheet CSV here (required before Section 3)
├── raw_fastq/                 ← downloaded raw FASTQs
├── fastqc_raw/                ← FastQC reports (raw)
├── trimmed_fastq/             ← TrimGalore output
├── fastqc_trimmed/            ← FastQC reports (trimmed)
├── multiqc_raw/               ← MultiQC (raw)
├── multiqc_trimmed/           ← MultiQC (trimmed)
├── genome/                    ← reference genome + GTF
├── star_index/                ← STAR genome index
├── rsem_index/                ← RSEM reference
├── aligned/                   ← STAR BAM output
├── counts/                    ← RSEM count tables
└── logs/                      ← all tool logs
```


---
## Section 0: System Benchmark

> **Overview:** RNA-seq analysis is computationally intensive. Before we start,
> we detect how many CPU cores and how much memory (RAM) your computer has. Many pipeline
> steps run in **parallel** (doing multiple samples at once), so knowing your hardware
> lets the pipeline automatically tune itself for maximum speed.
>
> - **CPU cores:** Each core can run one task at a time. More cores = more samples processed simultaneously.
> - **RAM:** Held in memory during processing. The genome alignment step (STAR) alone needs ~32 GB.
>
> This benchmark also checks that Python's multiprocessing is actually working on your system,
> and warns you if you don't have enough RAM for later steps.


In [ ]:
import os
import multiprocessing
import subprocess
import time
import math
from concurrent.futures import ProcessPoolExecutor
import platform

# ── Detect system resources ───────────────────────────────────────────────────
TOTAL_CORES    = multiprocessing.cpu_count()
USABLE_CORES   = max(1, TOTAL_CORES - 2)   # leave 2 cores free for OS

# RAM detection (Linux)
try:
    with open('/proc/meminfo') as f:
        mem_info = f.read()
    total_ram_kb = int([l for l in mem_info.split('\n') if 'MemTotal' in l][0].split()[1])
    total_ram_gb = total_ram_kb / (1024**2)
    free_ram_kb  = int([l for l in mem_info.split('\n') if 'MemAvailable' in l][0].split()[1])
    free_ram_gb  = free_ram_kb / (1024**2)
except:
    total_ram_gb = 0
    free_ram_gb  = 0

print("=" * 55)
print("  SYSTEM RESOURCES")
print("=" * 55)
print(f"  OS:               {platform.system()} {platform.release()}")
print(f"  Total CPU cores:  {TOTAL_CORES}")
print(f"  Usable cores:     {USABLE_CORES}  (total - 2 reserved for OS)")
print(f"  Total RAM:        {total_ram_gb:.1f} GB")
print(f"  Available RAM:    {free_ram_gb:.1f} GB")
print("=" * 55)


In [ ]:
# ── Parallel benchmark: measure actual multiprocessing speedup ────────────────
def cpu_task(n):
    """A CPU-bound task: compute sum of squares up to n."""
    return sum(i*i for i in range(n))

TASK_SIZE  = 5_000_000
N_TASKS    = USABLE_CORES * 2

print(f"Running benchmark: {N_TASKS} tasks of {TASK_SIZE:,} iterations each...")

# Serial
t0 = time.time()
for _ in range(N_TASKS):
    cpu_task(TASK_SIZE)
serial_time = time.time() - t0

# Parallel
t0 = time.time()
with ProcessPoolExecutor(max_workers=USABLE_CORES) as ex:
    list(ex.map(cpu_task, [TASK_SIZE] * N_TASKS))
parallel_time = time.time() - t0

speedup = serial_time / parallel_time

print(f"\n{'─'*45}")
print(f"  Serial time:    {serial_time:.2f}s")
print(f"  Parallel time:  {parallel_time:.2f}s  ({USABLE_CORES} cores)")
print(f"  Speedup:        {speedup:.2f}x")
print(f"{'─'*45}")

if speedup > 1.5:
    print(f"\n✅ Multiprocessing working well. Will use {USABLE_CORES} cores.")
else:
    print(f"\n⚠️  Speedup lower than expected. Check for resource contention.")
    print(f"   Will still proceed with {USABLE_CORES} cores.")

# ── STAR requires ~32 GB RAM for GRCm39; warn if insufficient ─────────────────
if total_ram_gb > 0 and total_ram_gb < 32:
    print(f"\n⚠️  WARNING: STAR genome indexing requires ~32 GB RAM.")
    print(f"   Your system has {total_ram_gb:.1f} GB total. This may fail.")
elif total_ram_gb >= 32:
    print(f"\n✅ RAM sufficient for STAR genome indexing (need ~32 GB).")


---
## Section 1: Tool Installation Check

> **Overview:** This pipeline relies on several specialized bioinformatics
> tools. Before we invest hours downloading data and running analyses, we verify every
> required program is installed and accessible. Each tool has a specific job:
>
> | Tool | Job |
> |---|---|
> | **FastQC** | Inspect raw sequencing read quality |
> | **TrimGalore / Cutadapt** | Remove adapter sequences from reads |
> | **STAR** | Align reads to the reference genome |
> | **RSEM** | Count how many reads map to each gene |
> | **SAMtools** | Manipulate alignment (BAM) files |
> | **MultiQC** | Combine QC reports from all samples |
> | **RSeQC** | Post-alignment quality assessment |
> | **wget** | Download files from the internet |
>
> If any tool shows ❌ MISSING, run the conda install command from the Prerequisites section above.


In [ ]:
# ── Check all required tools ───────────────────────────────────────────────────
REQUIRED_TOOLS = {
    'fastqc':       {'version_flag': '--version',  'geneLab_version': '0.12.1'},
    'trim_galore':  {'version_flag': '--version',  'geneLab_version': '0.6.10'},
    'cutadapt':     {'version_flag': '--version',  'geneLab_version': '4.2'},
    'STAR':         {'version_flag': '--version',  'geneLab_version': '2.7.11b'},
    'rsem-calculate-expression': {'version_flag': '--version', 'geneLab_version': '1.3.3'},
    'samtools':     {'version_flag': '--version',  'geneLab_version': '1.21'},
    'multiqc':      {'version_flag': '--version',  'geneLab_version': '1.26'},
    'infer_experiment.py': {'version_flag': '--version', 'geneLab_version': 'any'},
    'wget':         {'version_flag': '--version',  'geneLab_version': 'any'},
}

all_ok = True
print(f"{'Tool':<30} {'Status':<10} {'GeneLab Version'}")
print("-" * 60)
for tool, info in REQUIRED_TOOLS.items():
    try:
        result = subprocess.run([tool, info['version_flag']],
                                capture_output=True, text=True)
        output = (result.stdout + result.stderr).split('\n')[0][:30]
        print(f"{tool:<30} {'✅ found':<10} (GeneLab: {info['geneLab_version']})")
    except FileNotFoundError:
        print(f"{tool:<30} {'❌ MISSING':<10} (GeneLab: {info['geneLab_version']})")
        all_ok = False

print()
if all_ok:
    print("✅ All tools found. Ready to proceed.")
else:
    print("❌ Some tools missing. Run the conda install command in the Prerequisites section.")


---
## Section 2: Project Directory Setup & Configuration

> **Overview:** RNA-seq pipelines produce a lot of files, including raw data,
> quality reports, aligned reads, count tables, and logs. Creating a consistent
> folder structure from the start makes it much easier to find outputs and share
> your work with others.
>
> This section creates all the folders we'll need and saves a **config file**
> (`pipeline_config.json`) that the subsequent notebooks load automatically,
> so you only need to set your project path once.

> ⚙️ **Customize:** Change `PROJECT_ROOT` below to wherever you want the project stored.
> Use an absolute path with plenty of disk space (~500 GB needed for the full dataset).


In [ ]:
from pathlib import Path
import json

# ══════════════════════════════════════════════════════════════════
#  ← EDIT THIS: Set your desired project location
PROJECT_ROOT = Path.home() / "GLDS525_project"
# ══════════════════════════════════════════════════════════════════

# Define all subdirectories
DIRS = {
    'runsheet':       PROJECT_ROOT / 'runsheet',
    'raw_fastq':      PROJECT_ROOT / 'raw_fastq',
    'fastqc_raw':     PROJECT_ROOT / 'fastqc_raw',
    'trimmed_fastq':  PROJECT_ROOT / 'trimmed_fastq',
    'fastqc_trimmed': PROJECT_ROOT / 'fastqc_trimmed',
    'multiqc_raw':    PROJECT_ROOT / 'multiqc_raw',
    'multiqc_trimmed':PROJECT_ROOT / 'multiqc_trimmed',
    'genome':         PROJECT_ROOT / 'genome',
    'star_index':     PROJECT_ROOT / 'star_index',
    'rsem_index':     PROJECT_ROOT / 'rsem_index',
    'aligned':        PROJECT_ROOT / 'aligned',
    'counts':         PROJECT_ROOT / 'counts',
    'rseqc':          PROJECT_ROOT / 'rseqc',
    'logs':           PROJECT_ROOT / 'logs',
}

# Create all directories
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {'created' if not path.exists() else 'ok':8}  {path}")

# Save config for use in subsequent notebooks
config = {
    'PROJECT_ROOT': str(PROJECT_ROOT),
    'N_CORES': USABLE_CORES,
    'TOTAL_RAM_GB': round(total_ram_gb, 1),
    **{k: str(v) for k, v in DIRS.items()}
}
config_path = PROJECT_ROOT / 'pipeline_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ Project root: {PROJECT_ROOT}")
print(f"✅ Config saved: {config_path}")
print(f"   (Subsequent notebooks load this config automatically)")


---
## Section 3: Download Raw FASTQs from GeneLab

> **Overview:** A **FASTQ file** is the raw output from a DNA sequencer.
> It contains millions of short DNA sequences called **reads** (typically 150 base pairs
> long), each paired with a quality score for every base. Because we're doing
> **paired-end** sequencing, every sample has two files: R1 (the "forward" read) and
> R2 (the "reverse" read); together they cover more of each fragment.
>
> This study has **29 mouse samples**, each with R1 + R2 files, totalling **58 FASTQ
> files** and approximately **350–400 GB** of data. Downloads use `wget` with automatic
> resume support, so if your connection drops, you can re-run this cell and it will
> pick up where it left off.

> ⚙️ **Customize the settings below before running:**
>
> 1. **`RUNSHEET_PATH`**: Make sure the filename matches the CSV you downloaded from GeneLab
>    and placed in your `runsheet/` folder.  
>    Default: `GLDS-525_rna_seq_bulkRNASeq_v2_runsheet.csv`
>
> 2. **`TEST_MODE`**: Set `True` to download only the first sample (2 files) as a
>    connectivity test before committing to the full 400 GB download.
>
> 3. **`N_DOWNLOAD_WORKERS`**: How many files to download simultaneously. The default
>    (`min(4, USABLE_CORES)`) is a safe choice. Reduce to `1` or `2` on slower connections.


In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# ══════════════════════════════════════════════════════════════════
#  ← EDIT if your runsheet has a different filename
RUNSHEET_PATH = DIRS['runsheet'] / 'GLDS-525_rna_seq_bulkRNASeq_v2_runsheet.csv'

#  ← EDIT: Set True to download only the first sample (connectivity test)
TEST_MODE     = False

#  ← EDIT: Parallel download workers (reduce on slow connections)
N_DOWNLOAD_WORKERS = min(4, USABLE_CORES)
# ══════════════════════════════════════════════════════════════════

if not RUNSHEET_PATH.exists():
    print(f"⚠️  Runsheet not found at: {RUNSHEET_PATH}")
    print(f"   Please copy the runsheet CSV to: {DIRS['runsheet']}")
    print(f"   Download it from: https://genelab-data.ndc.nasa.gov/genelab/accession/GLDS-525/")
else:
    runsheet = pd.read_csv(RUNSHEET_PATH)
    print(f"Runsheet loaded: {len(runsheet)} samples")
    print(f"Columns: {runsheet.columns.tolist()}")
    display(runsheet[['Sample Name', 'read1_path', 'read2_path', 'Factor Value[Spaceflight]']].head(3))


In [ ]:
def build_download_list(runsheet, raw_fastq_dir):
    """
    Build a list of (sample_name, url, local_path) tuples from the runsheet.
    The local filename mirrors the GeneLab naming convention:
    GLDS-525_rna-seq_{SampleName}_R1_raw.fastq.gz
    """
    downloads = []
    for _, row in runsheet.iterrows():
        sample = row['Sample Name']
        for read_num, url_col in [('R1', 'read1_path'), ('R2', 'read2_path')]:
            url = row[url_col]
            # Extract filename from URL
            fname = url.split('file=')[-1] if 'file=' in url else url.split('/')[-1]
            local_path = raw_fastq_dir / fname
            downloads.append({
                'sample':     sample,
                'read':       read_num,
                'url':        url,
                'local_path': local_path,
                'filename':   fname
            })
    return downloads

downloads = build_download_list(runsheet, DIRS['raw_fastq'])

if TEST_MODE:
    downloads = downloads[:2]   # first sample only (R1 + R2)
    print(f"TEST MODE: downloading {len(downloads)} files only")
else:
    print(f"Full download: {len(downloads)} files across {len(runsheet)} samples")

# Show what will be downloaded
already_done = sum(1 for d in downloads if d['local_path'].exists())
to_download  = len(downloads) - already_done
print(f"Already downloaded: {already_done} files")
print(f"To download:        {to_download} files")
print(f"Download workers:   {N_DOWNLOAD_WORKERS}")


In [ ]:
print_lock = threading.Lock()

def download_file(item):
    """Download a single file using wget with resume support."""
    local = item['local_path']
    
    # Skip if already complete (rough check: file exists and size > 1GB)
    if local.exists() and local.stat().st_size > 1_000_000_000:
        with print_lock:
            print(f"  ⏭  SKIP  {item['filename']} (already exists)")
        return True, item['filename']
    
    cmd = [
        'wget',
        '--continue',          # resume if interrupted
        '--tries=5',           # retry up to 5 times
        '--timeout=60',
        '--quiet',
        '--show-progress',
        '--output-document', str(local),
        item['url']
    ]
    with print_lock:
        print(f"  ⬇  START {item['filename']}")
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    
    if result.returncode == 0:
        size_gb = local.stat().st_size / 1e9
        with print_lock:
            print(f"  ✅ DONE  {item['filename']}  ({size_gb:.2f} GB, {elapsed/60:.1f} min)")
        return True, item['filename']
    else:
        with print_lock:
            print(f"  ❌ FAIL  {item['filename']}  | {result.stderr[:100]}")
        return False, item['filename']


if to_download > 0:
    print(f"Starting parallel download ({N_DOWNLOAD_WORKERS} workers)...")
    pending = [d for d in downloads if not (d['local_path'].exists()
                                             and d['local_path'].stat().st_size > 1_000_000_000)]
    
    failed = []
    with ThreadPoolExecutor(max_workers=N_DOWNLOAD_WORKERS) as executor:
        futures = {executor.submit(download_file, d): d for d in pending}
        for future in as_completed(futures):
            ok, fname = future.result()
            if not ok:
                failed.append(fname)
    
    print(f"\nDownload complete.")
    if failed:
        print(f"❌ Failed files ({len(failed)}):")
        for f in failed:
            print(f"   {f}")
    else:
        print(f"✅ All {len(pending)} files downloaded successfully.")
else:
    print("✅ All files already downloaded.")


---
## Section 4: Raw Read Quality Control with FastQC

> **Overview:** Before we do anything with our data, we need to check its
> quality. **FastQC** examines each FASTQ file and generates a report covering:
>
> - **Per-base quality scores**: Are the bases called with high confidence?
> - **Adapter content**: Are there leftover sequencing adapter sequences?
> - **GC content**: Is the GC% what we'd expect for mouse (~50%)?
> - **Sequence duplication**: Is there unexpected overrepresentation of certain reads?
> - **Read length distribution**: Are all reads the expected 151 bp?
>
> We run FastQC on all 58 FASTQ files in parallel. Each file produces an HTML report
> and a zip archive. Don't worry if some metrics show "WARN" or "FAIL"; many are
> expected for RNA-seq data (e.g., high duplication from highly expressed genes).


In [ ]:
import glob

# Find all raw FASTQ files
raw_fastqs = sorted(DIRS['raw_fastq'].glob('*.fastq.gz'))
print(f"Found {len(raw_fastqs)} raw FASTQ files")

# Check which ones already have FastQC output
def fastqc_done(fastq_path, out_dir):
    stem = fastq_path.name.replace('.fastq.gz', '')
    return (out_dir / f"{stem}_fastqc.html").exists()

todo = [f for f in raw_fastqs if not fastqc_done(f, DIRS['fastqc_raw'])]
done = len(raw_fastqs) - len(todo)
print(f"Already QC'd: {done}  |  To process: {len(todo)}")


In [ ]:
def run_fastqc(fastq_path, out_dir, threads=2, memory=2048):
    """Run FastQC on a single FASTQ file."""
    log_file = DIRS['logs'] / f"fastqc_raw_{fastq_path.stem}.log"
    cmd = [
        'fastqc',
        '-o', str(out_dir),
        '-t', str(threads),
        '--memory', str(memory),
        str(fastq_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    log_file.write_text(result.stdout + result.stderr)
    return result.returncode == 0, fastq_path.name


if todo:
    # Run FastQC in parallel (one file per worker)
    # Use half the cores since FastQC uses 2 threads internally per job
    fastqc_workers = max(1, USABLE_CORES // 2)
    print(f"Running FastQC on {len(todo)} files ({fastqc_workers} parallel jobs)...")
    
    failed = []
    with ProcessPoolExecutor(max_workers=fastqc_workers) as ex:
        futures = [
            ex.submit(run_fastqc, f, DIRS['fastqc_raw'])
            for f in todo
        ]
        for i, future in enumerate(as_completed(futures), 1):
            ok, fname = future.result()
            status = '✅' if ok else '❌'
            print(f"  [{i:02d}/{len(todo)}] {status} {fname}")
            if not ok:
                failed.append(fname)
    
    if failed:
        print(f"\n❌ Failed: {failed}")
    else:
        print(f"\n✅ FastQC (raw) complete for all {len(todo)} files.")
else:
    print("✅ FastQC already run on all raw files.")


---
## Section 5: Aggregate Raw QC with MultiQC

> **Overview:** FastQC produced 58 separate HTML reports (one per file).
> **MultiQC** combines all of them into a single interactive report, making it easy
> to spot outlier samples at a glance.
>
> **What to look for in the MultiQC report:**
> - All samples should have similar read counts (~10–30 million reads each)
> - Per-base quality should be high (green zone) across most of the read length
> - Adapter content should be detectable but not overwhelming (TrimGalore will fix this)
> - Any sample that looks very different from the others may have a quality problem
>
> Open the saved HTML file in your browser to explore the interactive report.


In [ ]:
report_name = 'GLDS-525_rna_seq_raw_multiqc_GLbulkRNAseq'

cmd = [
    'multiqc',
    '--force',
    '--interactive',
    '-o', str(DIRS['multiqc_raw']),
    '-n', report_name,
    str(DIRS['fastqc_raw'])
]
print("Running MultiQC on raw FastQC reports...")
result = subprocess.run(cmd, capture_output=True, text=True)
(DIRS['logs'] / 'multiqc_raw.log').write_text(result.stdout + result.stderr)

report_path = DIRS['multiqc_raw'] / f"{report_name}.html"
if report_path.exists():
    print(f"✅ MultiQC report saved: {report_path}")
    print(f"   Open in browser: file://{report_path}")
else:
    print(f"❌ MultiQC failed. Check log: {DIRS['logs'] / 'multiqc_raw.log'}")
    print(result.stderr[:500])


---
## Section 6: Adapter Trimming with TrimGalore

> **Overview:** During sequencing library preparation, short synthetic
> DNA sequences called **adapters** are ligated to the ends of RNA fragments so the
> sequencer can recognize them. Sometimes the sequencer reads past the insert and into
> the adapter sequence, or reads low-quality bases at the 3' end.
>
> **TrimGalore** (which wraps the tool Cutadapt) removes these artifacts by:
> 1. **Adapter trimming**: detects and removes Illumina adapter sequences automatically
> 2. **Quality trimming**: cuts low-quality bases from the 3' end of reads
> 3. **Length filtering**: discards reads that become too short after trimming
>
> We use the **exact same parameters that GeneLab used**, ensuring our results are
> reproducible with their official processing. Output files are renamed to match
> GeneLab's naming convention.

**GeneLab parameters reproduced exactly:**
- `--gzip`: compress output
- `--cores 4`: Cutadapt threads per sample
- `--phred33`: standard Illumina quality encoding
- `--paired`: paired-end mode

Output files are renamed to match GeneLab's naming convention:
`{SampleName}_GLbulkRNAseq_R1_trimmed.fastq.gz`


In [ ]:
def get_sample_fastq_pairs(raw_fastq_dir, runsheet):
    """
    Build list of (sample_name, R1_path, R2_path) from runsheet and raw_fastq dir.
    """
    pairs = []
    for _, row in runsheet.iterrows():
        sample = row['Sample Name']
        # Extract filenames from download URLs
        r1_fname = row['read1_path'].split('file=')[-1]
        r2_fname = row['read2_path'].split('file=')[-1]
        r1_path  = raw_fastq_dir / r1_fname
        r2_path  = raw_fastq_dir / r2_fname
        
        if r1_path.exists() and r2_path.exists():
            pairs.append({'sample': sample, 'r1': r1_path, 'r2': r2_path})
        else:
            print(f"  ⚠️  Missing: {sample}  (R1: {r1_path.exists()}, R2: {r2_path.exists()})")
    return pairs

sample_pairs = get_sample_fastq_pairs(DIRS['raw_fastq'], runsheet)
print(f"Found {len(sample_pairs)} complete sample pairs (R1 + R2)")

def trimgalore_done(sample, trimmed_dir):
    r1_out = trimmed_dir / f"{sample}_GLbulkRNAseq_R1_trimmed.fastq.gz"
    r2_out = trimmed_dir / f"{sample}_GLbulkRNAseq_R2_trimmed.fastq.gz"
    return r1_out.exists() and r2_out.exists()

todo_trim = [p for p in sample_pairs if not trimgalore_done(p['sample'], DIRS['trimmed_fastq'])]
print(f"Already trimmed: {len(sample_pairs) - len(todo_trim)}  |  To trim: {len(todo_trim)}")


In [ ]:
def run_trimgalore(pair, trimmed_dir, logs_dir, cores=4):
    """
    Run TrimGalore on one paired-end sample, then rename output files
    to match GeneLab naming convention.
    """
    sample = pair['sample']
    r1     = pair['r1']
    r2     = pair['r2']
    
    # TrimGalore command (exact GeneLab parameters)
    cmd = [
        'trim_galore',
        '--gzip',
        '--cores', str(cores),
        '--phred33',
        '--paired',
        str(r1), str(r2),
        '--output_dir', str(trimmed_dir)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    log_path = logs_dir / f"trimgalore_{sample}.log"
    log_path.write_text(result.stdout + result.stderr)
    
    if result.returncode != 0:
        return False, sample, result.stderr[:200]
    
    # Rename output files to match GeneLab convention
    r1_stem = r1.name.replace('.fastq.gz', '')
    r2_stem = r2.name.replace('.fastq.gz', '')
    
    rename_map = {
        trimmed_dir / f"{r1_stem}_val_1.fq.gz":  trimmed_dir / f"{sample}_GLbulkRNAseq_R1_trimmed.fastq.gz",
        trimmed_dir / f"{r2_stem}_val_2.fq.gz":  trimmed_dir / f"{sample}_GLbulkRNAseq_R2_trimmed.fastq.gz",
        trimmed_dir / f"{r1_stem}.fastq.gz_trimming_report.txt": logs_dir / f"{sample}_R1_GLbulkRNAseq_trimming_report.txt",
        trimmed_dir / f"{r2_stem}.fastq.gz_trimming_report.txt": logs_dir / f"{sample}_R2_GLbulkRNAseq_trimming_report.txt",
    }
    for src, dst in rename_map.items():
        if src.exists():
            src.rename(dst)
    
    return True, sample, ''


if todo_trim:
    # TrimGalore uses 4 cores per sample; run samples in parallel
    trim_cores_per_sample = 4
    trim_workers = max(1, USABLE_CORES // trim_cores_per_sample)
    print(f"Trimming {len(todo_trim)} samples ({trim_workers} parallel, {trim_cores_per_sample} cores each)...")
    
    failed = []
    with ProcessPoolExecutor(max_workers=trim_workers) as ex:
        futures = [
            ex.submit(run_trimgalore, p, DIRS['trimmed_fastq'],
                      DIRS['logs'], trim_cores_per_sample)
            for p in todo_trim
        ]
        for i, future in enumerate(as_completed(futures), 1):
            ok, sample, err = future.result()
            status = '✅' if ok else '❌'
            print(f"  [{i:02d}/{len(todo_trim)}] {status} {sample}")
            if not ok:
                failed.append((sample, err))
    
    if failed:
        print(f"\n❌ Failed samples:")
        for s, e in failed:
            print(f"   {s}: {e}")
    else:
        print(f"\n✅ TrimGalore complete for all {len(todo_trim)} samples.")
else:
    print("✅ All samples already trimmed.")


---
## Section 7: Post-Trim FastQC and MultiQC

> **Overview:** After trimming, we run FastQC again to confirm the
> adapter removal worked. You should see:
>
> - **Adapter content** dropping to near zero
> - **Per-base quality** remaining high (possibly slightly lower at the very start and end)
> - **Sequence length distribution** now showing a slight spread below 151 bp (trimmed reads)
>
> Compare the before/after MultiQC reports side by side. If adapter content is still
> high, trimming parameters may need adjustment.


In [ ]:
# FastQC on trimmed reads
trimmed_fastqs = sorted(DIRS['trimmed_fastq'].glob('*_trimmed.fastq.gz'))
print(f"Found {len(trimmed_fastqs)} trimmed FASTQ files")

todo_fqc_trim = [f for f in trimmed_fastqs if not fastqc_done(f, DIRS['fastqc_trimmed'])]
print(f"FastQC already done: {len(trimmed_fastqs) - len(todo_fqc_trim)}  |  To process: {len(todo_fqc_trim)}")

if todo_fqc_trim:
    fastqc_workers = max(1, USABLE_CORES // 2)
    print(f"Running FastQC on {len(todo_fqc_trim)} trimmed files ({fastqc_workers} workers)...")
    
    failed = []
    with ProcessPoolExecutor(max_workers=fastqc_workers) as ex:
        futures = [
            ex.submit(run_fastqc, f, DIRS['fastqc_trimmed'])
            for f in todo_fqc_trim
        ]
        for i, future in enumerate(as_completed(futures), 1):
            ok, fname = future.result()
            status = '✅' if ok else '❌'
            print(f"  [{i:02d}/{len(todo_fqc_trim)}] {status} {fname}")
            if not ok:
                failed.append(fname)
    
    if not failed:
        print(f"\n✅ Post-trim FastQC complete.")
else:
    print("✅ Post-trim FastQC already complete.")


In [ ]:
# MultiQC on trimmed FastQC + trimming reports
report_name_trim = 'GLDS-525_rna_seq_trimmed_multiqc_GLbulkRNAseq'

cmd = [
    'multiqc',
    '--force',
    '--interactive',
    '-o', str(DIRS['multiqc_trimmed']),
    '-n', report_name_trim,
    str(DIRS['fastqc_trimmed']),
    str(DIRS['logs'])    # picks up trimming reports
]
print("Running MultiQC on trimmed reads...")
result = subprocess.run(cmd, capture_output=True, text=True)
(DIRS['logs'] / 'multiqc_trimmed.log').write_text(result.stdout + result.stderr)

report_path = DIRS['multiqc_trimmed'] / f"{report_name_trim}.html"
if report_path.exists():
    print(f"✅ MultiQC report saved: {report_path}")
    print(f"   Open in browser: file://{report_path}")
    print(f"\n→ Proceed to Notebook 2: Genome Indexing & Alignment")
else:
    print(f"❌ MultiQC failed. Check: {DIRS['logs'] / 'multiqc_trimmed.log'}")
